
# This notebook generates a list of top owners and operators in GOGET for a country, then uses AI to check for mergers and acquisitions

Method: Cobbled together by SZ on 2026 03 19 based on existing codebase, added the AI prompting based on directions from Googles "[Welcome to Colab](https://colab.research.google.com/#scrollTo=C1XfGyQXQK1n)"

In [44]:
# prepare and gather data
#run this to pull content from google drive
from google.colab import drive
drive.mount('/content/drive')

#Loading the libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime
from google.colab import ai
!pip install xlsxwriter

# name export path
export_gen_path = "/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/GOGET Iterations/Snapshots of GOGET main data set/"
export_qc_path = '/content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/'

# make timestamp
from datetime import datetime
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

print('this runs qc output will all be timestamped: ' + timestamp)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
this runs qc output will all be timestamped: 2026-03-19_21-10-17


In [27]:
# load in GOGET Field Export and Project Export
field_file_name = "goget-field-export-2026-03-19T155832.xlsx"
project_file_name= "goget-project-export-2026-03-19T164542.xlsx"
field_data = pd.read_excel(export_gen_path + field_file_name)
project_data = pd.read_excel(export_gen_path + project_file_name)

print(field_data.columns)
print(project_data.columns)

Index(['Project ID', 'Unit Name', 'Unit name local script', 'Fuel type',
       'Country', 'Subnational unit', 'Production Type', 'Status',
       'Status detail', 'Status year', 'Discovery year', 'FID Year',
       'Production start year', 'Operator', 'Owner(s)', 'Parent(s)',
       'Government unit ID', 'Wiki URL (project)', 'Wiki URL (field)',
       'Name Other', 'Latitude', 'Longitude', 'Location accuracy',
       'Onshore/Offshore', 'Field outline (WKT)', 'Basin', 'Block(s)'],
      dtype='object')
Index(['Project ID', 'Project Name', 'Project local script', 'Fuel type',
       'Country', 'Subnational unit', 'Production Type', 'Status',
       'Status detail', 'Status year', 'Discovery year', 'FID Year',
       'Production start year', 'Operator', 'Owner(s)', 'Parent(s)',
       'Government unit ID', 'Units (list of IDs)', 'Wiki URL (project)',
       'Name Other', 'Latitude', 'Longitude', 'Location accuracy',
       'Project location type', 'Onshore/Offshore', 'Project outline (

In [30]:
# define ownership columns
ownership_columns = ['Project ID', 'Country', 'Operator', 'Owner(s)', 'Parent(s)']

# filter to just ownership columns
GOGET_ownership_field = field_data[ownership_columns]
GOGET_ownership_project = project_data[ownership_columns]

#count rows in both projects and fields
print(GOGET_ownership_field.shape)
print(GOGET_ownership_project.shape)

# consolidate
GOGET_ownership = pd.concat([GOGET_ownership_field, GOGET_ownership_project])

print(GOGET_ownership.shape)

(1090, 5)
(138, 5)
(1228, 5)


In [31]:
# Explode 'Owner(s)' column by semicolon
GOGET_ownership_exploded = GOGET_ownership.assign(
    **{'Owner(s)': GOGET_ownership['Owner(s)'].str.split(';')} # wrap in list for explode
).explode('Owner(s)')

# Explode 'Parent(s)' column by semicolon on the already exploded DataFrame
GOGET_ownership_exploded = GOGET_ownership_exploded.assign(
    **{'Parent(s)': GOGET_ownership_exploded['Parent(s)'].str.split(';')} # wrap in list for explode
).explode('Parent(s)')

import re

# Function to remove ownership percentages
def remove_ownership_percents(text):
    if pd.isna(text):
        return text
    # Regex to find patterns like [xx.x%] or [x.x%] or [xx%] etc. and remove them along with any leading/trailing whitespace
    return re.sub(r'\s*\[\d+(\.\d+)?%\]', '', text).strip()

# Apply the function to 'Owner(s)' and 'Parent(s)' columns
GOGET_ownership_exploded['Owner(s)'] = GOGET_ownership_exploded['Owner(s)'].apply(remove_ownership_percents)
GOGET_ownership_exploded['Parent(s)'] = GOGET_ownership_exploded['Parent(s)'].apply(remove_ownership_percents)

In [36]:
GOGET_ownership_exploded

,Project ID,Country,Operator,Owner(s),Parent(s)
0,L100000321062,Uzbekistan,Surhan Gas Chemical Operating,Altmax Holding,Altmax Holding
0,L100000321062,Uzbekistan,Surhan Gas Chemical Operating,Altmax Holding,Gazprom PJSC
0,L100000321062,Uzbekistan,Surhan Gas Chemical Operating,Altmax Holding,Uzbekneftegaz JSC
0,L100000321062,Uzbekistan,Surhan Gas Chemical Operating,Gazprom PJSC,Altmax Holding
0,L100000321062,Uzbekistan,Surhan Gas Chemical Operating,Gazprom PJSC,Gazprom PJSC
...,...,...,...,...,...
135,L100000314099,Myanmar,PTTEP International,Myanmar Oil and Gas Enterprise,PTT Exploration and Production PCL
135,L100000314099,Myanmar,PTTEP International,Myanmar Oil and Gas Enterprise,Government of Myanmar
135,L100000314099,Myanmar,PTTEP International,Myanmar Oil and Gas Enterprise,Myanmar Oil and Gas Enterprise
136,L100000318348,Turkmenistan,Turkmengas State Concern,Turkmengas State Concern,Turkmengas State Concern


In [45]:
# break into 3 dataframes
Operator = GOGET_ownership_exploded[['Project ID', 'Country','Operator']]
Owner = GOGET_ownership_exploded[['Project ID', 'Country','Owner(s)']]
Parent = GOGET_ownership_exploded[['Project ID', 'Country','Parent(s)']]

In [57]:
# operator count by country
operator_counts = Operator.groupby(['Country', 'Operator'])['Project ID'].nunique().reset_index()
operator_counts.rename(columns={'Project ID': 'Asset Count'}, inplace=True)

top_10_operators_per_country = operator_counts.groupby('Country').apply(lambda x: x.nlargest(10, 'Asset Count')).reset_index(drop=True)

# owner count by country
owner_counts = Owner.groupby(['Country', 'Owner(s)'])['Project ID'].nunique().reset_index()
owner_counts.rename(columns={'Project ID': 'Asset Count'}, inplace=True)

top_10_owners_per_country = owner_counts.groupby('Country').apply(lambda x: x.nlargest(10, 'Asset Count')).reset_index(drop=True)

# parent count by country
parent_counts = Parent.groupby(['Country', 'Parent(s)'])['Project ID'].nunique().reset_index()
parent_counts.rename(columns={'Project ID': 'Asset Count'}, inplace=True)

top_10_parents_per_country = parent_counts.groupby('Country').apply(lambda x: x.nlargest(10, 'Asset Count')).reset_index(drop=True)

/tmp/ipykernel_268/704039443.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_10_operators_per_country = operator_counts.groupby('Country').apply(lambda x: x.nlargest(10, 'Asset Count')).reset_index(drop=True)
/tmp/ipykernel_268/704039443.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_10_owners_per_country = owner_counts.groupby('Country').apply(lambda x: x.nlargest(10, 'Asset Count')).rese

In [58]:
# export results to excel
output_excel_path = export_qc_path + 'top_ownership_data.xlsx'
writer = pd.ExcelWriter(output_excel_path, engine='xlsxwriter')

print(f"ExcelWriter object initialized for: {output_excel_path}")

top_10_operators_per_country.to_excel(writer, sheet_name='Operator', index=False)
print("Operator data written to 'Operator' sheet.")

top_10_owners_per_country.to_excel(writer, sheet_name='Owner', index=False)
print("Owner data written to 'Owner' sheet.")

top_10_parents_per_country.to_excel(writer, sheet_name='Parent', index=False)
print("Parent data written to 'Parent' sheet.")

writer.close()
print(f"Excel file saved at: {output_excel_path}")

ExcelWriter object initialized for: /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/top_ownership_data.xlsx
Operator data written to 'Operator' sheet.
Owner data written to 'Owner' sheet.
Parent data written to 'Parent' sheet.
Excel file saved at: /content/drive/Shareddrives/GEM Shared Drive/Programs/Oil & Gas Program/Global Oil and Gas Extraction Tracker (GOGET)/QC/QC Output Dump/top_ownership_data.xlsx


In [62]:
# unique countries in top_10_owners_per_country
unique_countries = top_10_owners_per_country['Country'].unique()

print(unique_countries)

['Azerbaijan' 'Bahrain' 'Bangladesh' 'Brunei' 'China' 'Cyprus' 'India'
 'Indonesia' 'Iran' 'Iraq' 'Israel' 'Japan' 'Jordan' 'Kazakhstan' 'Kuwait'
 'Malaysia' 'Mongolia' 'Myanmar' 'Oman' 'Pakistan' 'Palestine'
 'Philippines' 'Qatar' 'Saudi Arabia' 'Syria' 'Thailand' 'Timor-Leste'
 'Turkmenistan' 'Türkiye' 'United Arab Emirates' 'Uzbekistan' 'Vietnam'
 'Yemen']


In [63]:
# set country of interest
country_of_interest = 'Iran'

# filter operators to see top in set country
operators_in_country = top_10_operators_per_country[top_10_operators_per_country['Country'] == country_of_interest]

# Prepare a list to store AI responses
ai_check_results = []

# Iterate through each row in operators_in_country and query AI
for index, row in operators_in_country.iterrows():
    operator_name = row['Operator']
    AI_Merger_check = ai.generate_text("has" + operator_name + "had any recent mergers? please answer as briefly as possible")
    ai_check_results.append(AI_Merger_check)

# Add the AI responses as a new column to the DataFrame
operators_in_country['AI check'] = ai_check_results

print(operators_in_country)


   Country                             Operator  Asset Count  \
48    Iran                     Pars Oil and Gas           17   
49    Iran                 Iranian Offshore Oil           16   
50    Iran           Iranian Central Oil Fields           11   
51    Iran  South Zagros Oil and Gas Production           11   
52    Iran                 Arvandan Oil and Gas           10   
53    Iran      Aghajari Oil and Gas Production            7   
54    Iran     Gachsaran Oil and Gas Production            7   
55    Iran                 National Iranian Oil            7   
56    Iran           National Iranian South Oil            7   
57    Iran        Karoon Oil and Gas Production            6   

                                             AI check  
48                                                No.  
49                                                No.  
50                                                No.  
51                                                No.  
52             

/tmp/ipykernel_268/393273744.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  operators_in_country['AI check'] = ai_check_results
